In [1]:
import pandas as pd

In [2]:
melbourne_file_path = "data/melb_data.csv"
melbourne_data = pd.read_csv(melbourne_file_path)
melbourne_data.columns

Index(['Suburb', 'Address', 'Rooms', 'Type', 'Price', 'Method', 'SellerG',
       'Date', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car',
       'Landsize', 'BuildingArea', 'YearBuilt', 'CouncilArea', 'Lattitude',
       'Longtitude', 'Regionname', 'Propertycount'],
      dtype='object')

In [3]:
# Descartando as linhas com valores nulos
melbourne_data = melbourne_data.dropna(axis=0)

In [4]:
# Escolhendo o valor que queremos prever, por convenção nomeamos com y
y = melbourne_data.Price

In [5]:
# Definindo as 'características' que iremos utilizar para prever o valor de uma casa
melbourne_features = ["Rooms", "Bathroom", "Landsize", "Lattitude", "Longtitude"]

In [6]:
# Por convenção utilizamos o nome 'X' para definir os dados que iremos utilizar para prever 'y', igual numa função, onde existe uma dependência de y em x.
X: pd.DataFrame = melbourne_data[melbourne_features]
X.describe()

,Rooms,Bathroom,Landsize,Lattitude,Longtitude
count,6196.000000,6196.000000,6196.000000,6196.000000,6196.000000
mean,2.931407,1.576340,471.006940,-37.807904,144.990201
std,0.971079,0.711362,897.449881,0.075850,0.099165
min,1.000000,1.000000,0.000000,-38.164920,144.542370
25%,2.000000,1.000000,152.000000,-37.855438,144.926198
50%,3.000000,1.000000,373.000000,-37.802250,144.995800
75%,4.000000,2.000000,628.000000,-37.758200,145.052700
max,8.000000,8.000000,37000.000000,-37.457090,145.526350


In [7]:
X.head()

,Rooms,Bathroom,Landsize,Lattitude,Longtitude
1,2,1.0,156.0,-37.8079,144.9934
2,3,2.0,134.0,-37.8093,144.9944
4,4,1.0,120.0,-37.8072,144.9941
6,3,2.0,245.0,-37.8024,144.9993
7,2,1.0,256.0,-37.8060,144.9954


In [8]:
from sklearn.tree import DecisionTreeRegressor

melbourne_model = DecisionTreeRegressor(random_state=42)

melbourne_model.fit(X, y)

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [9]:
print("Fazendo previsões para as 5 casas seguintes:")
X.head()
print("As previsões são: ")
print(melbourne_model.predict(X.head()))

Fazendo previsões para as 5 casas seguintes:
As previsões são: 
[1035000. 1465000. 1600000. 1876000. 1636000.]


In [10]:
from sklearn.metrics import mean_absolute_error
# Quando utilizamos os mesmos dados que treinamos o modelo para verificar a assertividade do modelo, podemos ter a falsa sensação de que está dando bons resultados, já que o erro médio absoluto está relativamente baixo.
predicted_home_prices = melbourne_model.predict(X)
mean_absolute_error(y, predicted_home_prices)

1115.7467183128902

In [12]:
from sklearn.model_selection import train_test_split
# Agora estamos separando os nossos dados em duas partes: uma é a parte de dados para treinamento do nosso modelo e o outro é a parte para testes do modelo.
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=42)

melbourne_model = DecisionTreeRegressor()

melbourne_model.fit(train_X, train_y)

val_predictions = melbourne_model.predict(val_X)
# Agora que fizemos essa divisão dos dados em dois: dados de treinamento e dados de teste, podemos ver a diferença no erro absoluto médio, que está muito maior agora
mean_absolute_error(val_y, val_predictions)

253345.71530019367

In [13]:
# Muitas vezes o treinamento de um modelo passa por uma dessas fases: underfitting ou overfitting - o que acontece é que um modelo pode não estar aproveitando direito os dados, de maneira que sua precisão fica prejudicada para utilizar em novos dados, ou está tão bem treinada com os dados de maneira que ela não tenha aprendido chegar num resultado, mas sim decorado. Para isso precisamos saber e encontrar o meio-termo para não cairmos em nenhum desses dois casos.
def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
    model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=23)
    model.fit(train_X, train_y)
    preds_val = model.predict(val_X)
    mae = mean_absolute_error(val_y, preds_val)
    return mae

In [19]:
for max_leaf_nodes in [5, 50, 500, 5000]:
    mae = get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y)
    print(f"Max leaf nodes: {max_leaf_nodes} \t\t Mean Absolute Error: {mae}")

Max leaf nodes: 5 		 Mean Absolute Error: 369120.37021988904
Max leaf nodes: 50 		 Mean Absolute Error: 270711.29498949484
Max leaf nodes: 500 		 Mean Absolute Error: 242102.95697037815
Max leaf nodes: 5000 		 Mean Absolute Error: 254677.5638046051


In [20]:
from sklearn.ensemble import RandomForestRegressor

forest_model = RandomForestRegressor(random_state=23)
forest_model.fit(train_X, train_y)
melb_preds = forest_model.predict(val_X)
mean_absolute_error(val_y, melb_preds)

189613.75115722383